# 01 — Collecte institutionnelle : site web et PDF

Montre la découverte par sitemap, l'extraction HTML structurée, le téléchargement des PDF et la provenance enregistrée pour chaque document.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


## Entrées et sorties

- Entrées : sitemap et pages des domaines ENSEA.
- Sorties HTML/PDF extraites : JSON dans `raw/` et fichiers originaux dans `pdf/`.
- Métadonnées : URL, titre, titres de sections, liens, tableaux, dates et empreinte du contenu.


In [ ]:
import json
import pandas as pd
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
raw_files = sorted((config.project.data_path / 'raw').glob('*.json'))
rows = []
for path in raw_files[:200]:
    record = json.loads(path.read_text(encoding='utf-8'))
    rows.append({
        'url': record.get('url'),
        'title': record.get('title'),
        'characters': len(record.get('text', '')),
        'headings': len(record.get('headings', [])),
        'type': record.get('metadata', {}).get('doc_type', 'page'),
    })
print('Documents bruts présents :', len(raw_files))
display(pd.DataFrame(rows).head(10))


In [ ]:
from scraper import SitemapDiscoverer

RUN_SITEMAP_DISCOVERY = False
if RUN_SITEMAP_DISCOVERY:
    urls = SitemapDiscoverer(config.project.sitemap_url).discover()
    print('URLs découvertes :', len(urls))
    display(pd.Series(urls[:20], name='url').to_frame())
else:
    print('Découverte réseau désactivée. Passez RUN_SITEMAP_DISCOVERY à True pour la lancer.')


In [ ]:
from scraper import EnseaScraper

RUN_INSTITUTIONAL_COLLECTION = False
if RUN_INSTITUTIONAL_COLLECTION:
    await EnseaScraper(config).run()
    print('Collecte institutionnelle terminée dans', config.project.data_path)
else:
    print('Collecte réseau désactivée ; les données brutes déjà présentes seront réutilisées.')


## Contrôles attendus

Vérifiez la présence du texte principal, des dates lorsqu'elles existent, des URL permanentes et des PDF. Une collecte réussie ne signifie pas encore que le texte est propre : le nettoyage intervient au notebook 04.
